# 06_01_自己注意の_5_工程

コード：`ch06/blocks/06_01_自己注意の_5_工程.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 自己注意の 5 工程を「猫 が 魚 を 食べ た」で計算する（d=4）
import numpy as np

words = ["猫", "が", "魚", "を", "食べ", "た"]
# Q・K・V は結果が読めるよう手で決めた値（本物は学習で決まる）
# K の 1 列目＝名詞らしさ、2 列目＝動詞らしさ、3 列目＝助詞らしさ
K = np.array([[1.0, 0.0, 0.0, 0.2],    # 猫
              [0.0, 0.0, 1.0, 0.0],    # が
              [1.3, 0.2, 0.0, 0.0],    # 魚（食べ の目的語でもある）
              [0.0, 0.0, 1.0, 0.0],    # を
              [0.2, 1.0, 0.0, 0.0],    # 食べ
              [0.0, 0.6, 0.4, 0.0]])   # た
Q = np.array([[0.0, 0.0, 0.0, 1.0],    # 猫：手がかりなし
              [2.0, 0.0, 0.5, 0.0],    # が：直前の名詞を探す
              [0.0, 1.5, 0.0, 0.0],    # 魚：動詞を探す
              [2.0, 0.0, 0.5, 0.0],    # を：直前の名詞を探す
              [1.5, 0.0, 0.0, 0.0],    # 食べ：目的語を探す
              [0.0, 2.0, 0.0, 0.0]])   # た：動詞を探す
V = np.eye(6)          # 単位行列にすると、出力（⑤）が重みそのものになり読みやすい

def attention(q, K, V, mask=None, scale=True):
    s = K @ q                                   # ① 内積
    if scale:
        s = s / np.sqrt(len(q))                 # ② √d で割る
    if mask is not None:
        s = s + mask                            # ③ マスク／位置バイアス
    w = np.exp(s - s.max())
    w = w / w.sum()                             # ④ softmax
    return w @ V, w                             # ⑤ V の加重和

def eff(w):                                     # 実効的に見ている語数 exp(H)
    w = w[w > 0]
    return float(np.exp(-(w * np.log(w)).sum()))

n = len(words)
causal = np.where(np.arange(n) > np.arange(n)[:, None], -np.inf, 0.0)
posbias = -0.5 * np.abs(np.arange(n) - np.arange(n)[:, None])   # 遠い語ほど減点

def show(i, label, mask=None):
    _, w = attention(Q[i], K, V, None if mask is None else mask[i])
    cells = "  ".join(f"{words[j]}:{w[j]:.2f}" for j in range(n))
    print(f"{words[i]} {label:14s} {cells}   最大={words[w.argmax()]}  実効語数={eff(w):.2f}")

show(1, "マスクなし")
show(1, "因果マスク", causal)
show(1, "因果＋位置", causal + posbias)
show(3, "因果マスク", causal)
show(3, "因果＋位置", causal + posbias)
show(0, "因果マスク", causal)

# 「が」の行を工程ごとに表示する（因果マスクあり）
i = 1
s1 = K @ Q[i]; s2 = s1 / np.sqrt(4); s3 = s2 + causal[i]
w = np.exp(s3 - s3.max()); w = w / w.sum()
for j in range(n):
    print(f"  {words[j]:3s} ①{s1[j]:5.2f}  ②{s2[j]:5.2f}  ③{s3[j]:6.2f}  ④{w[j]:.3f}")

# √d で割らないと何が起きるか：ランダムな q・k（各成分は平均 0・分散 1）
rng = np.random.default_rng(0)
for d in (4, 64, 1024):
    q, Kr = rng.normal(size=d), rng.normal(size=(6, d))
    for scale in (True, False):
        _, w = attention(q, Kr, np.zeros((6, 1)), scale=scale)
        print(f"d={d:5d}  {'÷√d' if scale else '割らない':4s}  最大の重み={w.max():.3f}  実効語数={eff(w):.2f}")
